# 🎙 내 목소리로 23개 언어 말하기 · 0원

**Chatterbox Multilingual** (오픈소스 · MIT)로 내 목소리 그대로 영어·스페인어·일본어… 음성을 만들어요.
실험에서 13개 언어 모두 "내 목소리와 닮음" 1등이었던 모델이에요.

**준비물:** 내 목소리 10~30초 (조용한 곳에서, 말소리만)

**런타임 → 런타임 유형 변경 → T4 GPU**를 켜고, 위에서부터 차례로 실행하세요.
설명 페이지: https://www.aicitybuilders.com/dubbing

⚠️ **본인 목소리이거나, 허락받은 목소리만** 쓰세요.

In [5]:
# ① 설치 (2~4분)
#
# 빨간 글씨(ERROR: pip's dependency resolver ...)가 떠도 정상입니다.
# 설치가 끝나면 세션이 자동으로 한 번 다시 시작됩니다. 그다음 ② 번 셀부터 실행하세요.

!pip install -q chatterbox-tts "setuptools<81"

import os
FLAG = '/content/.voice_setup_done'
if not os.path.exists(FLAG):
    open(FLAG, 'w').close()
    print('설치 완료. 세션을 다시 시작합니다 — 잠시 뒤 ② 번 셀부터 실행하세요.')
    os.kill(os.getpid(), 9)
else:
    print('준비 완료 — ② 번 셀로 넘어가세요.')

준비 완료 — ② 번 셀로 넘어가세요.


In [1]:
# ② 목소리 엔진 불러오기 (1~2분, 처음 한 번만 모델을 내려받아요)
import os, sys, types

# 0. Numba 및 librosa 호환성 해결을 위한 NumPy 버전 다운그레이드 및 자동 런타임 재시작
try:
    import numpy as np
    if np.__version__ >= '2.2.0':
        raise ImportError
except (ImportError, AttributeError):
    print("Numba 호환 NumPy 2.1 버전을 설치 중입니다. 설치 완료 후 세션이 자동으로 재시작됩니다...")
    !pip install -q "numpy>=2.1.0,<2.2.0"
    import os
    os.kill(os.getpid(), 9)

# 1. torchvision 버전 충돌 및 InterpolationMode 에러 완벽 해결을 위한 패키지 재조정
try:
    from torchvision.transforms import InterpolationMode
except (ImportError, RuntimeError):
    print("torch/torchvision 호환 버전을 설치 중입니다. 약 1분 정도 소요됩니다...")
    # index-url을 제외하고 Colab 기본 패키지 매니저를 활용해 안전하게 최신 버전을 설치합니다.
    !pip install -q --force-reinstall torch torchvision torchaudio

# Python 3.13 대응 typing_extensions 설치 및 Unpack 바인딩
try:
    from typing_extensions import Unpack
except ImportError:
    !pip install -q "typing_extensions>=4.11.0"
    from typing_extensions import Unpack

# 표준 typing 모듈에 Unpack 강제 주입
import typing
typing.Unpack = Unpack
if not hasattr(typing, 'Unpack'):
    setattr(typing, 'Unpack', Unpack)

# transformers.processing_utils 모듈 사전 등록
processing_utils_mock = types.ModuleType('transformers.processing_utils')
processing_utils_mock.Unpack = Unpack
sys.modules['transformers.processing_utils'] = processing_utils_mock

# 패키지 설치
try:
    import tokenizers
    import transformers
except (ImportError, ModuleNotFoundError):
    !pip install -q "tokenizers>=0.20.0" "transformers>=4.45.0" "accelerate>=0.26.0"
    import transformers

# 3.13 Lazy Import 버그를 시스템 레벨에서 원천적으로 차단하기 위한 Proxy Wrapper 클래스 정의
from transformers.models.llama.modeling_llama import LlamaModel, LlamaConfig
from transformers.models.gpt2.modeling_gpt2 import GPT2Model, GPT2Config

class TransformersProxy(object):
    def __init__(self, origin):
        self.__dict__['_origin'] = origin
        self.__dict__['_custom'] = {
            'LlamaModel': LlamaModel,
            'LlamaConfig': LlamaConfig,
            'GPT2Model': GPT2Model,
            'GPT2Config': GPT2Config,
            'processing_utils': processing_utils_mock
        }
    def __getattr__(self, name):
        if name in self._custom:
            return self._custom[name]
        return getattr(self._origin, name)
    def __setattr__(self, name, value):
        setattr(self._origin, name, value)

# 전역 sys.modules 및 로컬 네임스페이스에 Proxy 등록
sys.modules['transformers'] = TransformersProxy(sys.modules['transformers'])
import transformers
transformers.LlamaModel = LlamaModel
transformers.LlamaConfig = LlamaConfig
transformers.GPT2Model = GPT2Model
transformers.GPT2Config = GPT2Config

import subprocess, torch, torchaudio
from IPython.display import Audio, display

# ③번 셀에서 사용할 경로 변수를 최상단에서 우선 정의합니다.
OUT = os.environ.get('DUB_OUT', '/content/voice')
os.makedirs(OUT, exist_ok=True)

from chatterbox.mtl_tts import ChatterboxMultilingualTTS, SUPPORTED_LANGUAGES

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
tts = ChatterboxMultilingualTTS.from_pretrained(device=DEVICE)
print(f'준비 완료 · {DEVICE} · 언어 {len(SUPPORTED_LANGUAGES)}개:', ', '.join(sorted(SUPPORTED_LANGUAGES)))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


Cangjie5_TC.json:   0%|          | 0.00/1.92M [00:00<?, ?B/s]

Downloading: "https://github.com/explosion/spacy-pkuseg/releases/download/v0.0.26/spacy_ontonotes.zip" to /root/.pkuseg/spacy_ontonotes.zip
100%|██████████| 34567143/34567143 [00:00<00:00, 113396299.11it/s]


loaded PerthNet (Implicit) at step 250,000
준비 완료 · cuda · 언어 23개: ar, da, de, el, en, es, fi, fr, he, hi, it, ja, ko, ms, nl, no, pl, pt, ru, sv, sw, tr, zh


In [2]:
# ③ 내 목소리 올리기 — 10~30초, 말소리만 깨끗하게
#
# 파일 선택 창이 뜨면 mp3 / wav / m4a / mp4 아무거나 올리세요.
if os.environ.get('DUB_LOCAL') == '1':
    src = os.environ['DUB_VOICE']
else:
    from google.colab import files
    src = next(iter(files.upload()))

VOICE = os.path.join(os.path.dirname(OUT), 'my_voice.wav')      # 결과 폴더와 따로 보관
subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', src, '-ac', '1', '-ar', '24000', '-t', '30', VOICE], check=True)   # 모노 · 24kHz · 앞 30초
display(Audio(VOICE))

Saving 내 목소리 녹음.m4a to 내 목소리 녹음 (4).m4a


In [4]:
# ④ 문장 하나 만들어 보기 — 글자와 언어만 바꾸면 돼요
TEXT = "Hi, I'm Jay from Connect AI Lab. This is my own voice, now speaking your language."
LANG = 'en'                 # en 영어 · es 스페인어 · fr 프랑스어 · hi 힌디어 … ②번에 나온 목록

wav = tts.generate(TEXT, language_id=LANG, audio_prompt_path=VOICE,
                   exaggeration=0.5,     # 감정 세기 0.25~1.0 (클수록 생생하게)
                   cfg_weight=0.5)       # 0.3 쯤으로 낮추면 말이 차분하고 느려져요
torchaudio.save(f'{OUT}/{LANG}.wav', wav.cpu(), tts.sr)
display(Audio(f'{OUT}/{LANG}.wav'))

/usr/lib/python3.13/contextlib.py:109: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
Sampling:  16%|█▋        | 164/1000 [00:06<00:33, 25.29it/s]


In [5]:
# ⑤ 여러 언어로 한 번에
SAY = {
    'en': "Hi, I'm Jay from Connect AI Lab. This is my own voice, now speaking your language.",
    'es': 'Hola, soy Jay de Connect AI Lab. Esta es mi propia voz, ahora hablando tu idioma.',
    'fr': "Bonjour, je suis Jay de Connect AI Lab. C'est ma propre voix, qui parle maintenant votre langue.",
    'hi': 'नमस्ते, मैं Connect AI Lab से जय हूँ। यह मेरी अपनी आवाज़ है, जो अब आपकी भाषा में बोल रही है।',
    'zh': '大家好，我是 Connect AI Lab 的 Jay。这是我自己的声音，现在正在说你的语言。',
}
for lang, text in SAY.items():
    wav = tts.generate(text, language_id=lang, audio_prompt_path=VOICE)
    torchaudio.save(f'{OUT}/{lang}.wav', wav.cpu(), tts.sr)
    print(lang, text); display(Audio(f'{OUT}/{lang}.wav'))

Sampling:  12%|█▏        | 123/1000 [00:04<00:35, 25.06it/s]


en Hi, I'm Jay from Connect AI Lab. This is my own voice, now speaking your language.


Sampling:  11%|█         | 108/1000 [00:04<00:38, 23.45it/s]


es Hola, soy Jay de Connect AI Lab. Esta es mi propia voz, ahora hablando tu idioma.


Sampling:  16%|█▌        | 162/1000 [00:06<00:34, 24.24it/s]


fr Bonjour, je suis Jay de Connect AI Lab. C'est ma propre voix, qui parle maintenant votre langue.


Sampling:  19%|█▉        | 189/1000 [00:07<00:33, 24.17it/s]


hi नमस्ते, मैं Connect AI Lab से जय हूँ। यह मेरी अपनी आवाज़ है, जो अब आपकी भाषा में बोल रही है।


Sampling:  17%|█▋        | 170/1000 [00:07<00:35, 23.16it/s]


zh 大家好，我是 Connect AI Lab 的 Jay。这是我自己的声音，现在正在说你的语言。


In [6]:
# ⑥ 내려받기 — 만든 음성을 zip 하나로
import shutil
shutil.make_archive(f'{OUT}_all', 'zip', OUT)
if os.environ.get('DUB_LOCAL') != '1':
    from google.colab import files
    files.download(f'{OUT}_all.zip')
print('완료 ✅', sorted(f for f in os.listdir(OUT) if f.endswith('.wav')))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

완료 ✅ ['en.wav', 'es.wav', 'fr.wav', 'hi.wav', 'zh.wav']


### 팁
- 같은 문장도 실행할 때마다 조금씩 달라요. 마음에 안 들면 그 셀만 다시 실행하세요.
- **일본어**는 한자를 잘 못 읽어요. 히라가나·가타카나로 바꿔 넣으면 훨씬 정확해져요.
- **인도네시아어·베트남어·태국어**는 이 모델이 지원하지 않아요.
- 영상에 쓰면 설명란에 "AI 음성(내 목소리)"이라고 적어 주세요.